In [15]:
import torch
import random
import numpy as np
import pandas as pd 
from torch.utils.data import Dataset, DataLoader

In [2]:
physio_path = '/nfs/turbo/umms-adraelos/caloric_restriction_DO_mice/Genotype_data/physio_data_standardized_nonan.csv'
physio_data = pd.read_csv(physio_path, index_col=False)
physio_data = physio_data.iloc[:, 1:]
physio_data

,MouseID,Generation,SurvDays,Y1A_BW_BW,Y2A_BW_BW,Y3A_BW_BW,Y1_Glu.F_Glucose,Y2_Glu.F_Glucose,Y3_Glu.F_Glucose,Y1_Echo_BPM,...,Y2_Wheel_AvgDistLFC,Y3_Wheel_AvgDistLFC,Y1A_Grip_All,Y2A_Grip_All,Y3A_Grip_All,Y1A_Frailty_FrailtyAdj,Y2A_Frailty_FrailtyAdj,Y3A_Frailty_FrailtyAdj,Diet_num,Diet
0,DO-40-2162,28,1611,-0.307435,-0.307056,-0.307631,-0.283439,-0.288608,-0.298913,-0.226083,...,-0.311391,-0.310923,-0.296258,-0.282809,-0.280893,-0.311680,-0.311663,-0.311658,1,40
1,DO-40-2179,28,1557,-0.306467,-0.307639,-0.307650,-0.284068,-0.299355,-0.299814,-0.226654,...,-0.311626,-0.311557,-0.261626,-0.280519,-0.275873,-0.311654,-0.311646,-0.311645,1,40
2,DO-40-2181,28,1533,-0.308128,-0.308415,-0.308165,-0.284068,-0.295729,-0.301298,-0.222349,...,-0.311650,-0.311652,-0.273628,-0.276688,-0.284453,-0.311674,-0.311646,-0.311652,1,40
3,DO-40-2187,28,1520,-0.307231,-0.306669,-0.306499,-0.289778,-0.297047,-0.299484,-0.231736,...,-0.311489,-0.311527,-0.286703,-0.273420,-0.275873,-0.311667,-0.311656,-0.311648,1,40
4,DO-40-2099,26,1638,-0.307528,-0.307634,-0.307756,-0.283129,-0.299220,-0.301403,-0.231980,...,-0.311381,-0.311312,-0.266797,-0.281854,-0.285387,-0.311667,-0.311659,-0.311637,1,40
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
932,DO-AL-0009,22,349,-0.308062,-0.300877,-0.300094,-0.287516,-0.267897,-0.266845,-0.227472,...,-0.312827,-0.312155,-0.268713,-0.300365,-0.301679,-0.311634,-0.311603,-0.311590,5,AL
933,DO-1D-3015,22,309,-0.306428,-0.300877,-0.300094,-0.285460,-0.267897,-0.266845,-0.194762,...,-0.312827,-0.312155,-0.283665,-0.300365,-0.301679,-0.311634,-0.311603,-0.311590,4,1D
934,DO-2D-4010,22,309,-0.308167,-0.300877,-0.300094,-0.286944,-0.267897,-0.266845,-0.194762,...,-0.312827,-0.312155,-0.281983,-0.300365,-0.301679,-0.311634,-0.311603,-0.311590,3,2D
935,DO-1D-3006,22,244,-0.307097,-0.300877,-0.300094,-0.292383,-0.267897,-0.266845,-0.194762,...,-0.312827,-0.312155,-0.293681,-0.300365,-0.301679,-0.311634,-0.311603,-0.311590,4,1D


In [27]:
np.min(physio_data.iloc[:, 1:-2]), np.max(physio_data.iloc[:, 3:-2])

(-0.3128270086048231, 7.107052564259018)

In [28]:
physio_var = physio_data.iloc[:, 1:-1].var(axis=0)
physio_var

Generation                4.644642e+00
SurvDays                  7.912356e+04
Y1A_BW_BW                 7.714110e-07
Y2A_BW_BW                 4.055867e-06
Y3A_BW_BW                 1.050198e-05
Y1_Glu.F_Glucose          3.907626e-05
Y2_Glu.F_Glucose          8.940893e-05
Y3_Glu.F_Glucose          1.783399e-04
Y1_Echo_BPM               1.126723e-04
Y2_Echo_BPM               1.575102e-04
Y3_Echo_BPM               1.259674e-04
Y1_Echo_CardiacOutput     1.251889e+00
Y2_Echo_CardiacOutput     2.253209e+00
Y3_Echo_CardiacOutput     3.134395e+00
Y1_CBC_Hgb                1.824325e-07
Y2_CBC_Hgb                7.084397e-07
Y3_CBC_Hgb                7.593765e-07
Y1_Rotarod_Mean           1.768308e-04
Y2_Rotarod_Mean           2.448031e-04
Y3_Rotarod_Mean           1.639053e-04
Y1_AS_MeanLog             1.928002e-09
Y2_AS_MeanLog             3.660646e-09
Y3_AS_MeanLog             8.982483e-10
Y1_Wheel_AvgSpeedLFC      5.653428e-08
Y2_Wheel_AvgSpeedLFC      2.549442e-08
Y3_Wheel_AvgSpeedLFC     

In [31]:
pd.DataFrame(physio_var.values).to_csv('/nfs/turbo/umms-adraelos/caloric_restriction_DO_mice/Genotype_data/physio_var.csv', index=False)

In [29]:
len(physio_var)

36

In [3]:
genoprobs_path = '/nfs/turbo/umms-adraelos/caloric_restriction_DO_mice/Genotype_data/genoprobs_proj.csv'
genoprobs = pd.read_csv(genoprobs_path)
genoprobs

,Unnamed: 0,0,1,2,3,4,5,6,7,8,...,5863,5864,5865,5866,5867,5868,5869,5870,5871,5872
0,DO-1D-3001,11.181311,12.383703,13.223754,18.683861,-3.105873,15.071812,4.566722,-8.317668,15.178740,...,-9.608385,1.688261,2.132044,6.027388,-1.819352,-14.918910,0.255752,-12.551320,-32.210761,-18.746747
1,DO-1D-3002,-2.628764,22.933458,7.645753,5.538203,7.347764,2.942720,11.347963,4.449972,-6.118222,...,7.449434,0.517254,19.643941,-1.064277,7.085058,-14.735193,3.369756,-20.902948,11.984909,-0.489731
2,DO-1D-3003,14.611459,-0.662600,-24.308638,-22.627848,-18.261186,1.859805,6.543754,22.646550,-1.809972,...,3.795336,8.341982,7.015957,-4.252234,17.039490,-25.052406,-2.387492,27.084163,16.082832,-7.912733
3,DO-1D-3004,-6.680621,-8.019135,3.125651,4.316594,11.230175,25.391169,2.784092,5.191860,0.504001,...,6.408988,-1.998942,5.332627,11.780933,0.869926,-2.390750,-14.677933,-7.229061,1.885609,-14.551324
4,DO-1D-3005,-7.762874,6.568555,24.752748,7.915682,8.565950,1.527536,-6.604914,12.055185,-5.745179,...,11.553945,4.265048,-6.110699,-3.763212,7.669412,-15.097044,-26.173256,31.715402,-14.482437,19.841158
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
941,DO-AL-0035,-2.603553,3.989595,5.078461,-12.081190,17.994081,1.477907,8.555104,15.169529,-11.607028,...,9.336485,-6.130255,1.278046,-5.116948,3.087162,-5.732800,0.773564,-15.927428,-3.684303,-8.617224
942,DO-AL-0064,-16.592022,-13.050886,-2.775860,6.993123,22.905117,-23.698464,-6.764692,3.005287,-21.190046,...,-24.039623,12.675260,40.292406,-15.214119,5.056168,15.422292,-13.980632,-3.702247,4.333789,6.259631
943,DO-AL-0073,37.893878,-8.674703,9.040660,12.313616,-18.294495,6.359014,-2.191113,-2.120036,-3.447605,...,29.586744,19.535960,18.001992,14.377207,9.018423,20.694376,24.442459,6.656219,-3.966847,7.416123
944,DO-AL-0089,-9.300763,-7.110903,-9.108345,0.429022,26.645567,-21.359485,10.953367,22.687536,10.444933,...,-13.349453,-0.512996,1.010019,7.261418,-33.304929,7.421048,18.690578,10.997129,2.577963,-22.869207


In [20]:
np.min(genoprobs.iloc[:, 1:]), np.max(genoprobs.iloc[:, 1:])

(-73.2288948863759, 64.73026584617132)

In [33]:
genoprobs_var = genoprobs.iloc[:, 1:].var(axis=0)
genoprobs_var

0       159.708584
1       142.847381
2       143.785389
3       165.711994
4       151.712388
           ...    
5868    170.743224
5869    143.170025
5870    160.085848
5871    142.560699
5872    149.217425
Length: 5873, dtype: float64

In [34]:
pd.DataFrame(genoprobs_var.values).to_csv('/nfs/turbo/umms-adraelos/caloric_restriction_DO_mice/Genotype_data/genoprobs_var.csv', index=False)

In [4]:
genoprobs = genoprobs.rename(columns={genoprobs.columns[0]: 'MouseID'})
genoprobs

,MouseID,0,1,2,3,4,5,6,7,8,...,5863,5864,5865,5866,5867,5868,5869,5870,5871,5872
0,DO-1D-3001,11.181311,12.383703,13.223754,18.683861,-3.105873,15.071812,4.566722,-8.317668,15.178740,...,-9.608385,1.688261,2.132044,6.027388,-1.819352,-14.918910,0.255752,-12.551320,-32.210761,-18.746747
1,DO-1D-3002,-2.628764,22.933458,7.645753,5.538203,7.347764,2.942720,11.347963,4.449972,-6.118222,...,7.449434,0.517254,19.643941,-1.064277,7.085058,-14.735193,3.369756,-20.902948,11.984909,-0.489731
2,DO-1D-3003,14.611459,-0.662600,-24.308638,-22.627848,-18.261186,1.859805,6.543754,22.646550,-1.809972,...,3.795336,8.341982,7.015957,-4.252234,17.039490,-25.052406,-2.387492,27.084163,16.082832,-7.912733
3,DO-1D-3004,-6.680621,-8.019135,3.125651,4.316594,11.230175,25.391169,2.784092,5.191860,0.504001,...,6.408988,-1.998942,5.332627,11.780933,0.869926,-2.390750,-14.677933,-7.229061,1.885609,-14.551324
4,DO-1D-3005,-7.762874,6.568555,24.752748,7.915682,8.565950,1.527536,-6.604914,12.055185,-5.745179,...,11.553945,4.265048,-6.110699,-3.763212,7.669412,-15.097044,-26.173256,31.715402,-14.482437,19.841158
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
941,DO-AL-0035,-2.603553,3.989595,5.078461,-12.081190,17.994081,1.477907,8.555104,15.169529,-11.607028,...,9.336485,-6.130255,1.278046,-5.116948,3.087162,-5.732800,0.773564,-15.927428,-3.684303,-8.617224
942,DO-AL-0064,-16.592022,-13.050886,-2.775860,6.993123,22.905117,-23.698464,-6.764692,3.005287,-21.190046,...,-24.039623,12.675260,40.292406,-15.214119,5.056168,15.422292,-13.980632,-3.702247,4.333789,6.259631
943,DO-AL-0073,37.893878,-8.674703,9.040660,12.313616,-18.294495,6.359014,-2.191113,-2.120036,-3.447605,...,29.586744,19.535960,18.001992,14.377207,9.018423,20.694376,24.442459,6.656219,-3.966847,7.416123
944,DO-AL-0089,-9.300763,-7.110903,-9.108345,0.429022,26.645567,-21.359485,10.953367,22.687536,10.444933,...,-13.349453,-0.512996,1.010019,7.261418,-33.304929,7.421048,18.690578,10.997129,2.577963,-22.869207


In [5]:
physio_data['MouseID'].isin(genoprobs['MouseID']).sum()

np.int64(929)

In [6]:
physio_data['MouseID'].dtypes

dtype('O')

In [7]:
# physio_data['MouseID'] = physio_data['MouseID'].astype(str)
# genoprobs['MouseID'] = genoprobs['MouseID'].astype(str)

In [8]:
genoprobs['MouseID'].dtypes

dtype('O')

In [9]:
physio_data['MouseID'].map(type).unique()

array([<class 'str'>], dtype=object)

In [10]:
genoprobs['MouseID'].map(type).unique()

array([<class 'str'>], dtype=object)

In [11]:
joined_genetic_physio = pd.merge(genoprobs, physio_data, on='MouseID', how='inner')
joined_genetic_physio

,MouseID,0,1,2,3,4,5,6,7,8,...,Y2_Wheel_AvgDistLFC,Y3_Wheel_AvgDistLFC,Y1A_Grip_All,Y2A_Grip_All,Y3A_Grip_All,Y1A_Frailty_FrailtyAdj,Y2A_Frailty_FrailtyAdj,Y3A_Frailty_FrailtyAdj,Diet_num,Diet
0,DO-1D-3001,11.181311,12.383703,13.223754,18.683861,-3.105873,15.071812,4.566722,-8.317668,15.178740,...,-0.311498,-0.312155,-0.292402,-0.284623,-0.287628,-0.311634,-0.311652,-0.311648,4,1D
1,DO-1D-3002,-2.628764,22.933458,7.645753,5.538203,7.347764,2.942720,11.347963,4.449972,-6.118222,...,-0.311658,-0.312155,-0.279973,-0.283755,-0.301679,-0.311634,-0.311630,-0.311590,4,1D
2,DO-1D-3003,14.611459,-0.662600,-24.308638,-22.627848,-18.261186,1.859805,6.543754,22.646550,-1.809972,...,-0.311330,-0.312155,-0.285493,-0.287892,-0.291662,-0.311634,-0.311653,-0.311646,4,1D
3,DO-1D-3004,-6.680621,-8.019135,3.125651,4.316594,11.230175,25.391169,2.784092,5.191860,0.504001,...,-0.312827,-0.312155,-0.285127,-0.287126,-0.301679,-0.311634,-0.311656,-0.311590,4,1D
4,DO-1D-3006,-23.280268,2.666684,1.877726,4.490820,14.846047,7.421518,-15.492281,6.287861,-14.073132,...,-0.312827,-0.312155,-0.293681,-0.300365,-0.301679,-0.311634,-0.311603,-0.311590,4,1D
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
924,DO-AL-0035,-2.603553,3.989595,5.078461,-12.081190,17.994081,1.477907,8.555104,15.169529,-11.607028,...,-0.312827,-0.312155,-0.288187,-0.286061,-0.301679,-0.311665,-0.311649,-0.311590,5,AL
925,DO-AL-0064,-16.592022,-13.050886,-2.775860,6.993123,22.905117,-23.698464,-6.764692,3.005287,-21.190046,...,-0.312827,-0.312155,-0.277084,-0.275552,-0.301679,-0.311670,-0.311634,-0.311590,5,AL
926,DO-AL-0073,37.893878,-8.674703,9.040660,12.313616,-18.294495,6.359014,-2.191113,-2.120036,-3.447605,...,-0.312827,-0.312155,-0.276792,-0.300365,-0.301679,-0.311668,-0.311603,-0.311590,5,AL
927,DO-AL-0089,-9.300763,-7.110903,-9.108345,0.429022,26.645567,-21.359485,10.953367,22.687536,10.444933,...,-0.312827,-0.312155,-0.291882,-0.300365,-0.301679,-0.311649,-0.311603,-0.311590,5,AL


In [12]:
#joined_genetic_physio.to_csv('/nfs/turbo/umms-adraelos/caloric_restriction_DO_mice/Genotype_data/joined_genetic_physio.csv', index=False)

In [13]:
joined_genetic_physio.iloc[:, 1:genetic_dims]

NameError: name 'genetic_dims' is not defined

In [ ]:
joined_genetic_physio.iloc[:, 5874:-2]

In [ ]:
class PairedDODataset(Dataset):
    def __init__(self, paired_df, genetic_dims, diet_col, id_col, label_col): 
        self.paired_df = paired_df
        self.genetic_dims = genetic_dims
        self.diet_col = diet_col 
        self.id_col = id_col
        self.label_col = label_col

    def __len__(self):
        return len(self.paired_df)

    def __getitem__(self, idx):
        sample = self.paired_df.iloc[idx]

        # for the genoprobs data 
        genetic_df = torch.tensor(sample[1:self.genetic_dims].values.astype('float32'))
        genetic_label = sample[self.label_col]
        genetic_diet = sample[self.diet_col]
        genetic_id = sample[self.id_col]

        # for the physiological data 
        physio_df = torch.tensor(sample[self.genetic_dims:-2].values.astype('float32'))
        physio_label = sample[self.label_col]
        physio_diet = sample[self.diet_col]
        physio_id = sample[self.id_col]

        return (genetic_df, genetic_label, genetic_diet, genetic_id), (physio_df, physio_label, physio_diet, physio_id)

In [ ]:
genetic_dims = 5874
diet_col = 'Diet'
id_col = 'MouseID'
label_col = 'Diet_num'
dataset = PairedDODataset(joined_genetic_physio, genetic_dims, diet_col, id_col, label_col)

In [ ]:
dataloader = DataLoader(dataset, batch_size=512, shuffle=True)

In [ ]:
first_iter = next(iter(dataloader))

In [ ]:
first_iter[0][0]

In [ ]:
first_iter[1][0].shape

In [ ]:
len(first_iter[1][3])